# Section 1 -- Inspecting an MCP server

**Before you run anything:** the MCP server must already be up.

```bash
./scripts/run_mcp_server.sh
```

We are going to look at the server the way an *agent* looks at it -- by asking
it what it can do, rather than by reading its source.

That is the whole idea of MCP. The agent does not import your code. It asks.

In [ ]:
from fastmcp import Client

# The URL is the entire integration contract. No SDK for our helpdesk, no
# generated client stubs, no shared types. Just an address.
MCP_URL = "http://127.0.0.1:8000/mcp/"

client = Client(MCP_URL)
print("client ready:", MCP_URL)

## 1. Discovery -- what tools exist?

`list_tools()` is the discovery call. Whatever comes back is what the model
will be offered.

In [ ]:
async with client:
    tools = await client.list_tools()

for t in tools:
    print(f"{t.name:24} {t.description.strip().splitlines()[0]}")

## 2. The schema -- and where it came from

Here is the part worth pausing on.

Look at `server.py`. We never wrote a JSON schema. We wrote a Python function
with type hints and a docstring. FastMCP produced everything below from that.

```python
@mcp.tool
def lookup_ticket(ticket_id: str) -> dict:
    """Look up a single support ticket by its id..."""
```

The type hints became the parameter types. The docstring became the description
the model reads to decide *when* to call it.

**Practical consequence for you as an FDE:** a vague docstring is a bug. It is
the prompt the model uses to choose this tool. Most "the agent didn't call my
tool" issues are docstring issues, not code issues.

In [ ]:
import json

lookup = next(t for t in tools if t.name == "lookup_ticket")

# FastMCP 3.x exposes this MCP wire field as `inputSchema` (camelCase, matching
# the raw JSON). FastMCP 4.x renames it to `input_schema`. We accept either, so
# this cell survives the upgrade -- and so you know the rename is coming.
schema = getattr(lookup, "inputSchema", None) or getattr(lookup, "input_schema")

print(json.dumps(schema, indent=2))


## 3. Calling a tool

Same shape a model would produce: a name, and a dict of arguments.

In [ ]:
async with client:
    result = await client.call_tool("lookup_ticket", {"ticket_id": "TICK-1001"})

print(result.data)

In [ ]:
# Search the knowledge base -- the other side of a support workflow.
async with client:
    result = await client.call_tool("search_knowledge_base", {"query": "vpn"})

for article in result.data:
    print(f"- {article['title']}")

## 4. The other two primitives

Tools get the attention, but MCP has three primitives and they differ by
**who decides to use them**:

| Primitive | Who controls it | Example here |
|---|---|---|
| Tool | the **model** decides | `lookup_ticket` |
| Resource | the **application** loads it | `helpdesk://tickets/open` |
| Prompt | the **user** picks it | `triage_prompt` |

A resource is read-only context. A prompt is a reusable, parameterised template
-- typically surfaced as a slash command or menu item in the client.

In [ ]:
async with client:
    resources = await client.list_resources()
    prompts = await client.list_prompts()

print("resources:", [str(r.uri) for r in resources])
print("prompts:  ", [p.name for p in prompts])

In [ ]:
async with client:
    content = await client.read_resource("helpdesk://tickets/open")

print(content[0].text)

## 5. Stateless HTTP -- the part that matters in production

Our server runs with `stateless_http=True`.

In the original MCP design, a client opened a session, got back an
`Mcp-Session-Id` header, and had to send that header on every subsequent
request. That means **every request in a conversation has to reach the same
process**.

For a customer running one server on a laptop: fine. For a customer deploying
behind a load balancer: a real problem.

And you cannot solve it with sticky sessions the usual way, because many MCP
clients issue requests with a plain `fetch()` and never send cookies back. The
load balancer has nothing to pin on.

Stateless mode removes the requirement. Below we drive the server with raw
HTTP and never send a session header at all -- and it still works.

> **Where this is heading:** MCP spec revision `2026-07-28` takes this further
> and removes sessions from the *protocol* itself, along with the `initialize`
> handshake. What we are running is the transport-level version of the same
> idea, available today on stable FastMCP. See `docs/04-stateless-mcp.md`.

In [ ]:
import httpx

# MCP over HTTP is just JSON-RPC 2.0 POSTed to one endpoint.
HEADERS = {
    "Content-Type": "application/json",
    # The server may reply as JSON or as a single SSE event, so accept both.
    "Accept": "application/json, text/event-stream",
}


def parse(response):
    """Return the JSON-RPC payload, whether it arrived as JSON or SSE."""
    body = response.text
    if body.lstrip().startswith("{"):
        return response.json()
    for line in body.splitlines():
        if line.startswith("data:"):
            return json.loads(line[5:].strip())
    return body


with httpx.Client() as http:
    init = http.post(MCP_URL, headers=HEADERS, json={
        "jsonrpc": "2.0", "id": 1, "method": "initialize",
        "params": {
            "protocolVersion": "2025-06-18",
            "capabilities": {},
            "clientInfo": {"name": "notebook", "version": "1.0"},
        },
    })

print("status:", init.status_code)
# The header that is NOT here is the point.
print("Mcp-Session-Id returned:", init.headers.get("mcp-session-id", "<none>"))

In [ ]:
# A completely separate request. No session header, no shared connection state.
# On a stateful server this would fail with "Missing session ID".
with httpx.Client() as http:
    call = http.post(MCP_URL, headers=HEADERS, json={
        "jsonrpc": "2.0", "id": 2, "method": "tools/call",
        "params": {"name": "lookup_ticket", "arguments": {"ticket_id": "TICK-1002"}},
    })

print("status:", call.status_code)
print(json.dumps(parse(call), indent=2)[:600])

**Why an FDE should care:** because that request carried no session, *any*
instance could have served it. Which means the customer's MCP server scales
with an ordinary round-robin load balancer -- no sticky sessions, no shared
session store, and it can run on serverless.

That is usually the first architecture question a customer asks, and now you
have the answer.

---

Next: `docs/02-lab-a2a.md`.